<a href="https://colab.research.google.com/github/Kaveesha-Vihanga/DS_Project/blob/component-3/Sentiment_Analysis_for_facilities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
from collections import Counter

def extract_colombo_area(loc_str):
    """
    Returns a group key like 'colombo_3' if area can be determined,
    otherwise returns None (meaning not a Colombo location or area unknown).
    Handles:
      - Strings containing "colombo" (e.g., "Colombo 00300", "Colombo 3")
      - Pure 5-digit postal codes that belong to Colombo (e.g., "00200" → colombo_2)
    """
    s = str(loc_str).strip()
    if not s:
        return None

    # Helper to map a 5-digit postal code to Colombo area
    def postal_to_area(code):
        if len(code) != 5 or not code.isdigit():
            return None
        # Pattern for areas 1-9: 00X00
        if code.startswith('00'):
            third = int(code[2])
            if 1 <= third <= 9:
                return f"colombo_{third}"
        # Pattern for areas 10-15: 0XY00 (XY = 10-15)
        if code.startswith('0') and code[3:5] == '00':
            second_two = int(code[1:3])
            if 10 <= second_two <= 15:
                return f"colombo_{second_two}"
        # Fallback: first two digits if they are 1-15 (e.g., 02000 -> 2)
        first_two = int(code[:2])
        if 1 <= first_two <= 15:
            return f"colombo_{first_two}"
        return None

    # CASE 1: String contains "colombo" (case-insensitive)
    if 'colombo' in s.lower():
        # Look for a number after "Colombo"
        match = re.search(r'colombo\s+(\d{1,5})\b', s, re.IGNORECASE)
        if match:
            num_str = match.group(1)
            # If it's a plain number 1-15, use directly
            if num_str.isdigit() and 1 <= int(num_str) <= 15:
                return f"colombo_{int(num_str)}"
            # If it's a 5-digit code, try to map
            if len(num_str) == 5:
                area = postal_to_area(num_str)
                if area:
                    return area
        # If no explicit number, look for any 5-digit postal code in the string
        postal_match = re.search(r'\b(\d{5})\b', s)
        if postal_match:
            area = postal_to_area(postal_match.group(1))
            if area:
                return area
        # If we have "colombo" but can't determine area, return None (keep raw)
        return None

    # CASE 2: String does NOT contain "colombo" – check if it's a pure 5-digit postal code
    # (i.e., after stripping, it's all digits and length 5)
    if s.replace(' ', '').isdigit() and len(s.replace(' ', '')) == 5:
        code = s.replace(' ', '')
        area = postal_to_area(code)
        if area:
            return area

    # Not a Colombo location or area unknown
    return None

# Build automatic mapping
raw_locations = df['raw_location'].tolist()
loc_to_group = {}
group_counter = Counter()

for raw in raw_locations:
    group = extract_colombo_area(raw)
    if group is None:
        group = raw  # keep raw string as group
    loc_to_group[raw] = group
    group_counter[group] += 1

# Manual overrides based on user feedback
manual_mappings = {
    'Colombo 02000': ('colombo_2', 'Colombo 00200'),          # 02000 → Colombo 2
    'Alwis Place': ('Moratuwa', 'Moratuwa'),                  # Alwis Place → Moratuwa
    'Dehiwala-Mount Lavinia': ('Dehiwala-Mount Lavinia 10370', 'Dehiwala-Mount Lavinia 10370'),  # bare → with postal
    '00200': ('colombo_2', 'Colombo 00200'),                   # pure postal → Colombo 2
    'Kotte 10100': ('Sri Jayawardenepura Kotte', 'Sri Jayawardenepura Kotte 10100')
     }

# Apply overrides to loc_to_group and adjust group_counter
for raw, (target_group, target_display) in manual_mappings.items():
    if raw in loc_to_group:
        old_group = loc_to_group[raw]
        loc_to_group[raw] = target_group
        group_counter[old_group] -= 1
        group_counter[target_group] += 1

# Determine display name for each group (most common raw string in that group)
raw_counts = df['raw_location'].value_counts().to_dict()
group_display = {}
for raw, group in loc_to_group.items():
    if group not in group_display:
        # Find the raw string with highest count among those belonging to this group
        best_raw = max((r for r, g in loc_to_group.items() if g == group), key=lambda r: raw_counts.get(r, 0))
        group_display[group] = best_raw

# Override display names for manually mapped groups
for raw, (target_group, target_display) in manual_mappings.items():
    group_display[target_group] = target_display

# Add columns to dataframe
df['location_group'] = df['raw_location'].map(loc_to_group)
df['location_display'] = df['location_group'].map(group_display)

# Print mapping for verification
print("\nNormalization mapping (sample of distinct raw → group):")
distinct_raw = df['raw_location'].unique()
for raw in sorted(distinct_raw)[:50]:
    group = loc_to_group[raw]
    display = group_display[group]
    print(f"  '{raw}' → group '{group}' → display '{display}'")

print("\nGroup counts (first 10):")
print(df['location_group'].value_counts().head(10))

print("\nDisplay name counts (first 10):")
print(df['location_display'].value_counts().head(10))

# ============================================================
# STEP 4: Define Facilities and Keywords
# ============================================================
facilities = {
    'AC': ['ac', 'air condition', 'a/c', 'aircon'],
    'gym': ['gym', 'fitness center', 'workout'],
    'wifi': ['wifi', 'internet', 'wireless', 'broadband'],
    'pool': ['pool', 'swimming', 'jacuzzi'],
    'breakfast': ['breakfast', 'buffet', 'dining'],
    'staff': ['staff', 'employee', 'reception', 'front desk', 'service'],
    'room': ['room', 'bed', 'bathroom', 'shower'],
    'cleanliness': ['clean', 'dirty', 'tidy', 'spotless'],
    'location': ['location', 'area', 'neighborhood', 'place'],
    'parking': ['parking', 'car park', 'garage']
}

# ============================================================
# STEP 5: Initialize Zero-Shot Classifier
# ============================================================
device = 0 if torch.cuda.is_available() else -1
print(f"\nUsing device: {'GPU' if device==0 else 'CPU'}")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
    batch_size=16
)

candidate_labels = ["positive", "negative", "neutral"]

# ============================================================
# STEP 6: Analyze Sentiment per Review per Facility
# ============================================================
# counts[location_group][facility] = {"pos":0, "neg":0, "neu":0}
counts = defaultdict(lambda: defaultdict(lambda: {"pos": 0, "neg": 0, "neu": 0}))

print("\nAnalyzing reviews... This may take a while depending on dataset size.")
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing reviews"):
    review = str(row['review_clean']) if pd.notna(row['review_clean']) else ""
    if not review:
        continue
    location_group = row['location_group']  # use normalized group key

    sentences = sent_tokenize(review)

    for facility, keywords in facilities.items():
        pattern = r'\b(?:' + '|'.join(re.escape(k) for k in keywords) + r')\b'
        relevant_sentences = [sent for sent in sentences if re.search(pattern, sent, re.IGNORECASE)]

        if not relevant_sentences:
            continue

        sentiments = []
        for sent in relevant_sentences:
            result = classifier(
                sent,
                candidate_labels=candidate_labels,
                hypothesis_template=f"The {facility} is {{}}."
            )
            sentiments.append(result['labels'][0])

        if sentiments:
            pos_count = sentiments.count('positive')
            neg_count = sentiments.count('negative')
            neu_count = sentiments.count('neutral')
            if pos_count >= neg_count and pos_count >= neu_count:
                overall = 'positive'
            elif neg_count >= pos_count and neg_count >= neu_count:
                overall = 'negative'
            else:
                overall = 'neutral'

            sentiment_map = {'positive': 'pos', 'negative': 'neg', 'neutral': 'neu'}
            key = sentiment_map[overall]
            counts[location_group][facility][key] += 1

# ============================================================
# STEP 7: Compute Percentages and Create Summary
# ============================================================
summary_rows = []
for location_group, fac_dict in counts.items():
    display_name = group_display.get(location_group, location_group)  # fallback
    for facility, sent_counts in fac_dict.items():
        pos = sent_counts['pos']
        neg = sent_counts['neg']
        neu = sent_counts['neu']
        total = pos + neg + neu
        if total == 0:
            continue
        pos_pct = (pos / total) * 100
        summary_rows.append({
            'location_display': display_name,
            'facility': facility,
            'positive_percentage': round(pos_pct, 1),
            'total_mentions': total
        })

# Moved this line here to ensure summary_rows_df is defined before its first use
summary_rows_df = pd.DataFrame(summary_rows)

# ============================================================
# STEP 8: Generate User-Friendly Output
# ============================================================
output_lines = []
for display_name in summary_rows_df['location_display'].unique():
    loc_data = summary_rows_df[summary_rows_df['location_display'] == display_name]
    facility_strings = [f"{row['facility']} {row['positive_percentage']}%"
                        for _, row in loc_data.iterrows()]
    line = f"{display_name}, " + ", ".join(facility_strings)
    output_lines.append(line)

# Save to Drive
output_dir = '/content/drive/MyDrive/DSGP Component 3'
os.makedirs(output_dir, exist_ok=True)
output_text = "\n".join(output_lines)
with open(f'{output_dir}/facility_satisfaction_summary.txt', 'w') as f:
    f.write(output_text)

# Also save detailed CSV
summary_rows_df.to_csv(f'{output_dir}/facility_satisfaction_detailed.csv', index=False)

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)
print(f"\nSummary saved to: {output_dir}/facility_satisfaction_summary.txt")
print(f"Detailed CSV saved to: {output_dir}/facility_satisfaction_detailed.csv")
print("\nPreview of summary:")
print(output_text[:500])



Normalization mapping (sample of distinct raw → group):
  '00200' → group 'colombo_2' → display 'Colombo 00200'
  'Alwis Place' → group 'Moratuwa' → display 'Moratuwa'
  'Battaramulla 10120' → group 'Battaramulla 10120' → display 'Battaramulla 10120'
  'Boralesgamuwa' → group 'Boralesgamuwa' → display 'Boralesgamuwa'
  'Colombo' → group 'Colombo' → display 'Colombo'
  'Colombo 00010' → group 'colombo_10' → display 'Colombo 00010'
  'Colombo 00100' → group 'colombo_1' → display 'Colombo 00100'
  'Colombo 00200' → group 'colombo_2' → display 'Colombo 00200'
  'Colombo 00300' → group 'colombo_3' → display 'Colombo 00300'
  'Colombo 00400' → group 'colombo_4' → display 'Colombo 00400'
  'Colombo 00500' → group 'colombo_5' → display 'Colombo 00500'
  'Colombo 00600' → group 'colombo_6' → display 'Colombo 00600'
  'Colombo 00700' → group 'colombo_7' → display 'Colombo 00700'
  'Colombo 00800' → group 'colombo_8' → display 'Colombo 00800'
  'Colombo 00900' → group 'colombo_9' → display 'Colo

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


Analyzing reviews... This may take a while depending on dataset size.


Processing reviews: 100%|██████████| 16252/16252 [24:44<00:00, 10.95it/s]



ANALYSIS COMPLETE

Summary saved to: /content/drive/MyDrive/DSGP Component 3/facility_satisfaction_summary.txt
Detailed CSV saved to: /content/drive/MyDrive/DSGP Component 3/facility_satisfaction_detailed.csv

Preview of summary:
Dehiwala-Mount Lavinia 10350, cleanliness 95.1%, location 90.1%, staff 91.4%, room 70.1%, breakfast 65.8%, gym 100.0%, pool 100.0%, parking 38.5%, AC 50.0%, wifi 8.3%
Colombo 00500, pool 79.5%, breakfast 78.3%, staff 89.8%, room 76.2%, cleanliness 91.6%, location 93.1%, wifi 44.8%, AC 66.7%, parking 40.0%
Sri Jayawardenepura Kotte 10100, pool 95.6%, staff 98.3%, room 98.1%, location 97.5%, breakfast 96.4%, cleanliness 100.0%, breakfast 83.3%, cleanliness 89.4%, room 73.6%, staff 93.5%, location 
